# COS30049 A2 - Classifier test metrics over training seeds and a bootstrap of the test set (Kaggle CPU session)

Runs `python -m explorer.cli class-distribution` (`explorer/classify_distribution.py`): every reported classifier on
Elliptic (temporal split) and BitcoinHeist (temporal and random schemes), with its tuned settings and validation
threshold, refitted with 10 seeds and its test scores bootstrapped 1,000 times. Elliptic comes from the attached
Kaggle dataset (its licence forbids sharing processed rows); BitcoinHeist is downloaded from UCI. CPU only.

**Settings:** Accelerator **None**, **Internet on**. **Input:** dataset `ellipticco/elliptic-data-set`.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "028ec4c2209f784ec2abba49cc1a5f73a59e7cd6"   # the code this run uses
PINS = ["numpy==2.2.6", "pandas==2.2.3", "scikit-learn==1.6.1", "scipy==1.17.1", "pyarrow==20.0.0", "joblib==1.4.2"]

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

In [ ]:
raw = SRC / "data" / "external" / "elliptic"
raw.mkdir(parents=True, exist_ok=True)
for name in ("elliptic_txs_features.csv", "elliptic_txs_classes.csv", "elliptic_txs_edgelist.csv"):
    shutil.copy(find_input(name), raw / name)
sh(sys.executable, "-m", "explorer.cli", "external", "--bitcoinheist", log="external.log")
sh(sys.executable, "-m", "explorer.cli", "process-external", log="process-external.log")
sh(sys.executable, "-m", "explorer.cli", "class-distribution", log="class-distribution.log")
tables = WORK / "reports" / "tables"
tables.mkdir(parents=True, exist_ok=True)
for name in ("stats_classifiers_seeds.csv", "stats_classifiers_summary.csv", "stats_classifiers_paired.csv"):
    shutil.copy(SRC / "reports" / "tables" / name, tables / name)
(WORK / "kaggle_run.json").write_text(json.dumps({"commit": COMMIT, "hours": round((time.time() - T0) / 3600, 2),
    "packages": {p.split("==")[0]: version(p.split("==")[0]) for p in PINS}}, indent=2), encoding="utf-8")